<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_09_Real_FHIR_to_Candidate_OMOP_ETL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from collections import Counter
import os, sys, json, hashlib, hmac, secrets, uuid, sqlite3, zipfile, io, urllib.request, platform, re
import pandas as pd
import jsonschema

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

BASE=Path('/content')
PROJECT=BASE/'drive/MyDrive/NeuroFHIR_ICHI2027'
PRIVATE=BASE/'ICHI2027_Phase09_PRIVATE_SESSION'
PRIVATE.mkdir(parents=True,exist_ok=True)
OUT=PROJECT/'phase09_outputs'
OUT.mkdir(parents=True,exist_ok=True)
REPORT=OUT/'ICHI2027_Phase09_SHAREABLE.json'
SOURCE_NAME='TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv'
FHIR_ZIP_URL='https://hl7.org/fhir/R4/fhir.schema.json.zip'
PINNED_R4_SCHEMA_SHA256='75e5560da3cf503895a44c8ca7af17a83b4cca6c2cb5ba1883d2aec0d1cb5ac6'

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):
            h.update(b)
    return h.hexdigest()

def now(): return datetime.now(timezone.utc).isoformat()

def canon(raw):
    v=Decimal(str(raw).strip())
    if not v.is_finite(): raise ValueError('Nonfinite measurement')
    return format(v.normalize(),'f')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
prior_paths=[PROJECT/'phase08_outputs/ICHI2027_Phase08_SHAREABLE.json',BASE/'ICHI2027_Phase08_SHAREABLE.json']
prior_file=next((p for p in prior_paths if p.is_file()),None)
if prior_file is None:
    from google.colab import files
    uploaded=files.upload()
    if 'ICHI2027_Phase08_SHAREABLE.json' not in uploaded:
        raise FileNotFoundError('ICHI2027_Phase08_SHAREABLE.json')
    prior_file=PRIVATE/'ICHI2027_Phase08_SHAREABLE.json'
    prior_file.write_bytes(uploaded['ICHI2027_Phase08_SHAREABLE.json'])
prior=json.loads(prior_file.read_text())
assert prior['phase']=='08_real_private_evidence_staging_and_vocabulary_crosswalk'
assert prior['gates']['real_private_staging_parity_verified'] is True
PRIOR_SHA=sha256_file(prior_file)

In [ ]:
source_options=[BASE/SOURCE_NAME,BASE/'inputs'/SOURCE_NAME,PROJECT/'inputs'/SOURCE_NAME,PROJECT/'source_data'/SOURCE_NAME]
source_path=next((p for p in source_options if p.is_file()),None)
if source_path is None:
    from google.colab import files
    uploaded=files.upload()
    if SOURCE_NAME not in uploaded: raise FileNotFoundError(SOURCE_NAME)
    source_path=PRIVATE/SOURCE_NAME
    source_path.write_bytes(uploaded[SOURCE_NAME])
data=pd.read_csv(source_path,dtype=str,keep_default_na=False,encoding='utf-8-sig')
data.columns=[c.strip() for c in data.columns]
FEATURES={
'LHIPPOC':('hippocampus','left'),'RHIPPOC':('hippocampus','right'),
'LENTORHIN':('entorhinal_cortex','left'),'RENTORHIN':('entorhinal_cortex','right'),
'LMIDTEMP':('middle_temporal','left'),'RMIDTEMP':('middle_temporal','right'),
'VENTRICLES':('ventricles','unspecified'),'ICV':('intracranial_volume','not_applicable'),
}
REQUIRED={'RID','PTID'}|{f'{t}_{s}' for t in ('BL','M12') for s in (
'VISCODE','EXAMDATE','IMAGEUID','LONIUID','VERSION','OVERALLQC','TEMPQC','VENTQC',*FEATURES)}
missing=sorted(REQUIRED-set(data.columns))
if missing: raise ValueError('Missing verified source columns: '+repr(missing))
if len(data)!=prior['input_schema']['rows']: raise ValueError('Phase08 input row count differs; stop and re-audit')
data['RID']=data['RID'].str.strip()
if data['RID'].eq('').any() or data['RID'].duplicated().any(): raise ValueError('RID integrity failure')
if data['PTID'].str.strip().eq('').any(): raise ValueError('PTID integrity failure')
for t in ('BL','M12'):
    dates=pd.to_datetime(data[f'{t}_EXAMDATE'],errors='coerce')
    if dates.isna().any(): raise ValueError('Missing/invalid source acquisition dates')
    if (data[f'{t}_IMAGEUID'].str.strip().eq('') & data[f'{t}_LONIUID'].str.strip().eq('')).any():
        raise ValueError('No image identifiers available')
    data['_parsed_'+t]=dates
if (data['_parsed_M12']<=data['_parsed_BL']).any(): raise ValueError('Invalid chronological order')
for t in ('BL','M12'):
    for feature in FEATURES:
        raw=data[f'{t}_{feature}'].str.strip()
        for value in raw[raw.ne('')].unique(): canon(value)

In [ ]:
PRIVATE_REPRO=PROJECT/'_private_reproducibility'
PRIVATE_REPRO.mkdir(parents=True,exist_ok=True)
KEY_FILE=PRIVATE_REPRO/'ichi2027_pseudonymization_key.hex'
KEY_REUSED=KEY_FILE.is_file()
if KEY_REUSED:
    KEY=bytes.fromhex(KEY_FILE.read_text(encoding='ascii').strip())
    if len(KEY)!=32: raise ValueError('Invalid existing private pseudonymization key')
else:
    KEY=secrets.token_bytes(32)
    KEY_FILE.write_text(KEY.hex(),encoding='ascii')
    try: KEY_FILE.chmod(0o600)
    except OSError: pass
def opaque(*parts):
    b='|'.join(map(str,parts)).encode('utf-8')
    return hmac.new(KEY,b,hashlib.sha256).hexdigest()
def full_url(*parts): return 'urn:uuid:'+str(uuid.UUID(hex=opaque(*parts)[:32]))
def local_id(prefix,*parts): return prefix+'-'+opaque(*parts)[:30]

FHIR_SRC_SYSTEM={
    'IMAGEUID':'https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/source-identifier/adni-imageuid',
    'LONIUID':'https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/source-identifier/adni-loniuid',
}
FHIR_FIELD_SYSTEM='https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/source-field'
BUNDLES=[]
EVIDENCE={}
source_event_counts=Counter()
for _,record in data.iterrows():
    rid=record['RID']
    patient_url=full_url('Patient',rid)
    patient={'resourceType':'Patient','id':local_id('p','Patient',rid),
             'identifier':[{'system':'https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/local-pseudonym',
                            'value':opaque('research-only-pseudonym',rid)}]}
    entries=[{'fullUrl':patient_url,'resource':patient}]
    for tp in ('BL','M12'):
        dt=record['_parsed_'+tp].date().isoformat()
        image_url=full_url('ImagingStudy',rid,tp)
        img_ids=[{'system':FHIR_SRC_SYSTEM[code], 'value':str(record[f'{tp}_{code}']).strip()}
                 for code in ('IMAGEUID','LONIUID') if str(record[f'{tp}_{code}']).strip()]
        img={'resourceType':'ImagingStudy','id':local_id('im','ImagingStudy',rid,tp),
             'status':'unknown','subject':{'reference':patient_url},
             'started':dt,'identifier':img_ids}
        entries.append({'fullUrl':image_url,'resource':img})
        for feature,(region,side) in FEATURES.items():
            src_col=f'{tp}_{feature}'
            raw=str(record[src_col]).strip()
            if not raw: continue
            dec=canon(raw)
            if abs(Decimal(dec))>Decimal('1e100'): raise ValueError('FHIR numeric magnitude unsupported')
            obs_url=full_url('Observation',rid,tp,feature)
            anatomy=(' '.join((side,region)) if side in ('left','right') else region)
            observation={
               'resourceType':'Observation','id':local_id('o','Observation',rid,tp,feature),
               'status':'unknown','category':[{'coding':[{'system':'http://terminology.hl7.org/CodeSystem/observation-category', 'code':'imaging'}]}],
               'identifier':[{'system':FHIR_FIELD_SYSTEM,'value':src_col}],
               'code':{'text':anatomy+' volume'},
               'subject':{'reference':patient_url},'effectiveDateTime':dt,
               'derivedFrom':[{'reference':image_url}],'bodySite':{'text':anatomy},
               'valueQuantity':{'value':float(Decimal(dec))},
            }
            entries.append({'fullUrl':obs_url,'resource':observation})
            EVIDENCE[obs_url]={'source_decimal':dec,'source_col':src_col,
                               'person_url':patient_url,'image_url':image_url}
            source_event_counts[src_col]+=1
    BUNDLES.append({'resourceType':'Bundle','type':'collection','entry':entries})
if sum(source_event_counts.values())!=prior['private_real_source_evidence_staging']['feature_events']:
    raise ValueError('Measurement count differs from Phase08')

In [ ]:
SCHEMA_STATUS={'download':'NOT_RUN','checksum_verified':False,'validation':'NOT_RUN','validated':{},'errors':{}}
try:
    request=urllib.request.Request(FHIR_ZIP_URL,headers={'User-Agent':'ICHI2027-FHIR-OMOP-research/1.0'})
    with urllib.request.urlopen(request,timeout=60) as f: rawzip=f.read()
    found_sha=hashlib.sha256(rawzip).hexdigest()
    SCHEMA_STATUS.update(download='RETRIEVED',schema_sha256=found_sha)
    if found_sha!=PINNED_R4_SCHEMA_SHA256:
        raise ValueError('Official R4 schema archive checksum differs from verified Phase04 snapshot')
    SCHEMA_STATUS['checksum_verified']=True
    archive=zipfile.ZipFile(io.BytesIO(rawzip))
    names=[n for n in archive.namelist() if n.endswith('fhir.schema.json')]
    if len(names)!=1: raise ValueError('R4 archive schema not uniquely found')
    spec=json.loads(archive.read(names[0]))
    definitions=spec['definitions']
    validators={}
    for typ in ('Bundle','Patient','ImagingStudy','Observation'):
        wrapper={'$schema':spec.get('$schema','http://json-schema.org/draft-06/schema#'),
                 'definitions':definitions,'$ref':'#/definitions/'+typ}
        validators[typ]=jsonschema.validators.validator_for(wrapper)(wrapper)
    validated=Counter(); failed=Counter(); first_fail={}
    for bundle in BUNDLES:
        for kind,res in [('Bundle',bundle)]+[(item['resource']['resourceType'],item['resource']) for item in bundle['entry']]:
            found=next(validators[kind].iter_errors(res),None)
            validated[kind]+=1
            if found is not None:
                failed[kind]+=1
                if kind not in first_fail:
                    first_fail[kind]={'validator':str(found.validator),
                                      'schema_path':list(map(str,found.schema_path)),
                                      'instance_path':list(map(str,found.path))}
    SCHEMA_STATUS.update(validation=('PASS' if not failed else 'FAIL'),validated=dict(validated),
                         errors=dict(failed),first_failure_paths=first_fail)
except Exception as e:
    SCHEMA_STATUS['validation']='UNAVAILABLE_OR_FAILED'
    SCHEMA_STATUS['error_type']=type(e).__name__
if SCHEMA_STATUS['validation']!='PASS':
    raise RuntimeError('Official FHIR R4 schema validation was not verified; no ETL mapping will be claimed: '+json.dumps(SCHEMA_STATUS))

In [ ]:
DB=PRIVATE/'real_fhir_to_omop_CANDIDATE_only.sqlite'
if DB.exists(): DB.unlink()
con=sqlite3.connect(DB)
con.execute('PRAGMA foreign_keys=ON')
con.executescript("""
CREATE TABLE PERSON_CANDIDATE (
 person_id INTEGER PRIMARY KEY, private_patient_key TEXT NOT NULL UNIQUE
);
CREATE TABLE IMAGE_OCCURRENCE_CANDIDATE (
 image_occurrence_id INTEGER PRIMARY KEY, person_id INTEGER NOT NULL REFERENCES PERSON_CANDIDATE(person_id),
 visit_occurrence_id INTEGER, procedure_occurrence_id INTEGER,
 wadors_uri TEXT, local_path TEXT, image_occurrence_date TEXT NOT NULL,
 image_study_UID TEXT, image_series_UID TEXT, modality_concept_id INTEGER,
 anatomic_site_concept_id INTEGER,
 private_imaging_key TEXT NOT NULL UNIQUE, private_timepoint TEXT NOT NULL
);
CREATE TABLE IMAGE_FEATURE_CANDIDATE (
 image_feature_id INTEGER PRIMARY KEY, person_id INTEGER NOT NULL REFERENCES PERSON_CANDIDATE(person_id),
 image_occurrence_id INTEGER NOT NULL REFERENCES IMAGE_OCCURRENCE_CANDIDATE(image_occurrence_id),
 image_feature_concept_id INTEGER, image_feature_type_concept_id INTEGER,
 image_feature_event_field_concept_id INTEGER, image_feature_event_id INTEGER,
 image_finding_concept_id INTEGER, image_finding_id INTEGER,
 anatomic_site_concept_id INTEGER, alg_system TEXT, alg_datetime TEXT,
 private_observation_key TEXT NOT NULL UNIQUE, source_column TEXT NOT NULL, laterality TEXT NOT NULL,
 region_text TEXT NOT NULL
);
CREATE TABLE MEASUREMENT_CANDIDATE (
 private_measurement_id INTEGER PRIMARY KEY, person_id INTEGER NOT NULL REFERENCES PERSON_CANDIDATE(person_id),
 private_image_feature_id INTEGER NOT NULL UNIQUE REFERENCES IMAGE_FEATURE_CANDIDATE(image_feature_id),
 measurement_concept_id INTEGER, measurement_date TEXT NOT NULL,
 value_as_number REAL NOT NULL, unit_concept_id INTEGER,
 private_source_value_decimal TEXT NOT NULL, private_roundtrip_equal INTEGER NOT NULL
);
CREATE TABLE PRIVATE_SOURCE_IDENTIFIER_LEDGER (
 image_occurrence_id INTEGER PRIMARY KEY REFERENCES IMAGE_OCCURRENCE_CANDIDATE(image_occurrence_id),
 imageuid_hmac TEXT, loniuid_hmac TEXT
);
""")

FHIR_KEY_INDEX={}
REF_ERRORS=Counter()
NEXT_OCC=0
NEXT_FEATURE=0
PERSON_URL_TO_ID={}
IMAGE_URL_TO_ID={}
OBS_SEEN=set()
for bundle in BUNDLES:
    items=bundle['entry']
    local={entry['fullUrl']:entry['resource'] for entry in items}
    if len(local)!=len(items): raise ValueError('Duplicate FHIR fullUrl within bundle')
    patients=[(u,r) for u,r in local.items() if r['resourceType']=='Patient']
    if len(patients)!=1: raise ValueError('Unexpected patient cardinality')
    patient_url,patient=patients[0]
    pid=len(PERSON_URL_TO_ID)+1
    if patient_url in PERSON_URL_TO_ID: raise ValueError('Duplicate patient URL')
    PERSON_URL_TO_ID[patient_url]=pid
    con.execute('INSERT INTO PERSON_CANDIDATE VALUES (?,?)',(pid,opaque('private_patient_key',patient_url)))
    for img_url,img in [(u,r) for u,r in local.items() if r['resourceType']=='ImagingStudy']:
        if img['subject']['reference']!=patient_url: REF_ERRORS['image_wrong_subject']+=1; continue
        NEXT_OCC+=1
        tp='BL' if img['started']==data.iloc[pid-1]['_parsed_BL'].date().isoformat() else 'M12'
        if img['started'] not in (data.iloc[pid-1]['_parsed_BL'].date().isoformat(),data.iloc[pid-1]['_parsed_M12'].date().isoformat()):
            REF_ERRORS['image_date_not_in_source']+=1;continue
        IMAGE_URL_TO_ID[img_url]=NEXT_OCC
        con.execute("""INSERT INTO IMAGE_OCCURRENCE_CANDIDATE VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)""",(
            NEXT_OCC,pid,None,None,None,None,img['started'],None,None,None,None,
            opaque('image_key',img_url),tp
        ))
        src_ident={x['system']:x['value'] for x in img.get('identifier',[])}
        con.execute('INSERT INTO PRIVATE_SOURCE_IDENTIFIER_LEDGER VALUES (?,?,?)',(
            NEXT_OCC,opaque('IMAGEUID',src_ident[FHIR_SRC_SYSTEM['IMAGEUID']]) if FHIR_SRC_SYSTEM['IMAGEUID'] in src_ident else None,
            opaque('LONIUID',src_ident[FHIR_SRC_SYSTEM['LONIUID']]) if FHIR_SRC_SYSTEM['LONIUID'] in src_ident else None,
        ))
    for obs_url,obs in [(u,r) for u,r in local.items() if r['resourceType']=='Observation']:
        if obs_url in OBS_SEEN: raise ValueError('Duplicate FHIR Observation URL')
        OBS_SEEN.add(obs_url)
        if obs['subject']['reference']!=patient_url: REF_ERRORS['observation_wrong_subject']+=1;continue
        image_ref=obs['derivedFrom'][0]['reference']
        if image_ref not in IMAGE_URL_TO_ID: REF_ERRORS['observation_unresolved_imaging']+=1;continue
        field_ids=[x['value'] for x in obs.get('identifier',[]) if x['system']==FHIR_FIELD_SYSTEM]
        if len(field_ids)!=1: REF_ERRORS['observation_unresolved_source_field']+=1;continue
        field=field_ids[0]
        if not re.fullmatch(r'(BL|M12)_(LHIPPOC|RHIPPOC|LENTORHIN|RENTORHIN|LMIDTEMP|RMIDTEMP|VENTRICLES|ICV)',field):
            REF_ERRORS['invalid_source_feature_code']+=1;continue
        tp,feature=field.split('_',1)
        imageid=IMAGE_URL_TO_ID[image_ref]
        occ_date=con.execute('SELECT image_occurrence_date,private_timepoint FROM IMAGE_OCCURRENCE_CANDIDATE WHERE image_occurrence_id=?',(imageid,)).fetchone()
        if obs['effectiveDateTime']!=occ_date[0] or tp!=occ_date[1]:
            REF_ERRORS['observation_date_or_timepoint_mismatch']+=1;continue
        evidence=EVIDENCE.get(obs_url)
        if evidence is None or evidence['source_col']!=field: REF_ERRORS['source_lineage_mismatch']+=1;continue
        if evidence['image_url']!=image_ref or evidence['person_url']!=patient_url:
            REF_ERRORS['source_reference_mismatch']+=1;continue
        source_decimal=Decimal(evidence['source_decimal'])
        fhir_decimal=Decimal(str(obs['valueQuantity']['value']))
        numeric_equal=(fhir_decimal==source_decimal)
        NEXT_FEATURE+=1
        region,side=FEATURES[feature]
        con.execute("""INSERT INTO IMAGE_FEATURE_CANDIDATE VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",(
            NEXT_FEATURE,pid,imageid,None,None,None,None,None,None,None,None,None,
            opaque('feature_key',obs_url),field,side,region
        ))
        con.execute("""INSERT INTO MEASUREMENT_CANDIDATE VALUES (?,?,?,?,?,?,?,?,?)""",(
            NEXT_FEATURE,pid,NEXT_FEATURE,None,obs['effectiveDateTime'],float(obs['valueQuantity']['value']),
            None,evidence['source_decimal'],int(numeric_equal)
        ))
con.commit()
if REF_ERRORS: raise ValueError('Actual FHIR -> candidate mapping failed referential checks: '+repr(dict(REF_ERRORS)))

In [ ]:
# Execute duplicate-import idempotence check using the exact same FHIR keys, without inserting new rows.
before={table:con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0] for table in (
    'PERSON_CANDIDATE','IMAGE_OCCURRENCE_CANDIDATE','IMAGE_FEATURE_CANDIDATE','MEASUREMENT_CANDIDATE')}
for obs in OBS_SEEN:
    con.execute('INSERT OR IGNORE INTO IMAGE_FEATURE_CANDIDATE (image_feature_id,person_id,image_occurrence_id,private_observation_key,source_column,laterality,region_text) SELECT image_feature_id,person_id,image_occurrence_id,private_observation_key,source_column,laterality,region_text FROM IMAGE_FEATURE_CANDIDATE WHERE private_observation_key=?',(opaque('feature_key',obs),))
con.commit()
after={table:con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0] for table in before}
assert before==after
foreign_key_errors=len(con.execute('PRAGMA foreign_key_check').fetchall())
assert foreign_key_errors==0
count_mismatches={
    'patients':len(data)-before['PERSON_CANDIDATE'],
    'imaging_events':2*len(data)-before['IMAGE_OCCURRENCE_CANDIDATE'],
    'feature_events':len(EVIDENCE)-before['IMAGE_FEATURE_CANDIDATE'],
    'measurements':len(EVIDENCE)-before['MEASUREMENT_CANDIDATE'],
}
if any(count_mismatches.values()): raise ValueError('Missing or duplicate mapping records')
exact_value_mismatches=con.execute('SELECT COUNT(*) FROM MEASUREMENT_CANDIDATE WHERE private_roundtrip_equal=0').fetchone()[0]
source_membership={opaque('private_patient_key',full_url('Patient',rid)) for rid in data['RID']
    if all(str(data.loc[data['RID']==rid, f'{tp}_{f}'].iloc[0]).strip()
           for tp in ('BL','M12') for f in ('LHIPPOC','RHIPPOC'))}
target_membership={r[0] for r in con.execute("""
SELECT p.private_patient_key FROM PERSON_CANDIDATE p
JOIN IMAGE_OCCURRENCE_CANDIDATE i ON i.person_id=p.person_id
JOIN IMAGE_FEATURE_CANDIDATE f ON f.image_occurrence_id=i.image_occurrence_id
WHERE f.source_column IN ('BL_LHIPPOC','BL_RHIPPOC','M12_LHIPPOC','M12_RHIPPOC')
GROUP BY p.person_id HAVING COUNT(DISTINCT f.source_column)=4 AND COUNT(DISTINCT i.private_timepoint)=2
""")}
structural_diff=len(source_membership.symmetric_difference(target_membership))
if structural_diff: raise ValueError('Structural cohort FHIR-to-target membership mismatch')

In [ ]:
# Optional Athena scan. Candidates are vocabulary records, never approved mappings.
vocab_roots=[PROJECT/'vocabulary',BASE/'vocabulary']
vocab_files=[]
for folder in vocab_roots:
    if folder.is_dir():
        vocab_files+=list(folder.glob('*.zip'))+list(folder.glob('CONCEPT.csv'))+list(folder.glob('VOCABULARY.csv'))
vocab_files=list(dict.fromkeys(p.resolve() for p in vocab_files))
VOCAB={'snapshot_found':False,'snapshot_sha256':None,'vocabulary_versions':[],
       'concept_records_scanned':0,'candidate_concepts':[], 'status':'NO_ATHENA_SNAPSHOT'}
archives=[p for p in vocab_files if p.suffix.lower()=='.zip']
if archives:
    archive_path=max(archives,key=lambda x:x.stat().st_mtime)
    archive=zipfile.ZipFile(archive_path)
    concept_names=[n for n in archive.namelist() if n.upper().endswith('CONCEPT.CSV')]
    vocab_names=[n for n in archive.namelist() if n.upper().endswith('VOCABULARY.CSV')]
    if len(concept_names)==1 and len(vocab_names)==1:
        VOCAB['snapshot_found']=True
        VOCAB['snapshot_sha256']=sha256_file(archive_path)
        concept_stream=archive.open(concept_names[0])
        vocabulary_stream=archive.open(vocab_names[0])
    else:
        VOCAB['status']='ZIP_MISSING_REQUIRED_FULL_CONCEPT_AND_VOCABULARY'
else:
    concept_paths=[p for p in vocab_files if p.name.upper()=='CONCEPT.CSV']
    vocabulary_paths=[p for p in vocab_files if p.name.upper()=='VOCABULARY.CSV']
    if concept_paths and vocabulary_paths:
        VOCAB['snapshot_found']=True
        VOCAB['snapshot_sha256']=hashlib.sha256((sha256_file(concept_paths[0])+sha256_file(vocabulary_paths[0])).encode()).hexdigest()
        concept_stream=open(concept_paths[0],'rb')
        vocabulary_stream=open(vocabulary_paths[0],'rb')

if VOCAB['snapshot_found']:
    import csv
    def detect_sep(stream):
        peek=stream.readline().decode('utf-8-sig','replace')
        stream.seek(0)
        return '\t' if peek.count('\t')>=peek.count(',') else ','
    sep=detect_sep(vocabulary_stream)
    for batch in pd.read_csv(vocabulary_stream,sep=sep,dtype=str,keep_default_na=False,
                             encoding='utf-8-sig',chunksize=50000):
        if {'vocabulary_id','vocabulary_version'}.issubset(batch.columns):
            for pair in batch[['vocabulary_id','vocabulary_version']].itertuples(index=False,name=None):
                if any(k in pair[0].lower() for k in ('snomed','loinc','cpt','omop','dicom','ucum')):
                    VOCAB['vocabulary_versions'].append({'vocabulary_id':pair[0],'version':pair[1]})
    vocabulary_stream.close()
    sep=detect_sep(concept_stream)
    needles=('magnetic resonance','hippocamp','entorhinal','middle temporal','intracranial','cerebral ventricle','imaging feature')
    for batch in pd.read_csv(concept_stream,sep=sep,dtype=str,keep_default_na=False,
                             encoding='utf-8-sig',chunksize=100000):
        if not {'concept_id','concept_name','domain_id','standard_concept','invalid_reason','vocabulary_id'}.issubset(batch.columns):
            VOCAB['status']='CONCEPT_REQUIRED_COLUMNS_MISSING'
            break
        VOCAB['concept_records_scanned']+=len(batch)
        relevant=batch[batch['concept_name'].str.lower().str.contains('|'.join(needles),regex=True,na=False)]
        relevant=relevant[(relevant['standard_concept']=='S') & (relevant['invalid_reason']=='')]
        for _,r in relevant.head(300).iterrows():
            if len(VOCAB['candidate_concepts'])<80:
                VOCAB['candidate_concepts'].append({k:r[k] for k in (
                    'concept_id','concept_name','domain_id','standard_concept','vocabulary_id')})
    concept_stream.close()
    if VOCAB['status']=='NO_ATHENA_SNAPSHOT': VOCAB['status']='CANDIDATE_ONLY_NOT_APPROVED'

In [ ]:
blocked_columns={table:con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0] for table in ('IMAGE_OCCURRENCE_CANDIDATE','IMAGE_FEATURE_CANDIDATE')}
missing_mandatory={
    'image_occurrence_missing_procedure_occurrence_id':con.execute('SELECT COUNT(*) FROM IMAGE_OCCURRENCE_CANDIDATE WHERE procedure_occurrence_id IS NULL').fetchone()[0],
    'image_occurrence_missing_verified_study_uid':con.execute('SELECT COUNT(*) FROM IMAGE_OCCURRENCE_CANDIDATE WHERE image_study_UID IS NULL').fetchone()[0],
    'image_occurrence_missing_verified_series_uid':con.execute('SELECT COUNT(*) FROM IMAGE_OCCURRENCE_CANDIDATE WHERE image_series_UID IS NULL').fetchone()[0],
    'image_occurrence_missing_approved_modality_concept':con.execute('SELECT COUNT(*) FROM IMAGE_OCCURRENCE_CANDIDATE WHERE modality_concept_id IS NULL').fetchone()[0],
    'image_feature_missing_approved_concept':con.execute('SELECT COUNT(*) FROM IMAGE_FEATURE_CANDIDATE WHERE image_feature_concept_id IS NULL').fetchone()[0],
    'image_feature_missing_approved_type_concept':con.execute('SELECT COUNT(*) FROM IMAGE_FEATURE_CANDIDATE WHERE image_feature_type_concept_id IS NULL').fetchone()[0],
    'measurement_missing_unit_concept':con.execute('SELECT COUNT(*) FROM MEASUREMENT_CANDIDATE WHERE unit_concept_id IS NULL').fetchone()[0],
}
PRIVATE_MANIFEST={
    'phase':'09', 'run_utc':now(),
    'source_csv_sha256_private':sha256_file(source_path),
    'phase08_sha256':PRIOR_SHA,
    'fhir_schema_sha256':SCHEMA_STATUS.get('schema_sha256'),
    'hmac_key_reused_from_private_reproducibility_store':KEY_REUSED,
    'source_rows':len(data),
    'candidate_etl_counts':before,
    'foreign_key_failures':foreign_key_errors,
}
PRIVATE_MANIFEST_FILE=PRIVATE_REPRO/'phase09_private_reproducibility_manifest.json'
PRIVATE_MANIFEST_FILE.write_text(json.dumps(PRIVATE_MANIFEST,indent=2),encoding='utf-8')
REPORT_DATA={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'09_actual_fhir_to_private_omop_candidate_etl',
 'run_utc':now(),
 'result_status':'ACTUAL_FHIR_TO_PRIVATE_CANDIDATE_TRANSFORMATION_ONLY_NOT_OFFICIAL_MI_CDM_LOAD_NOT_FOUR_ARM_COMPARISON',
 'privacy':'No source rows, participant identifiers, raw FHIR bundles or private database exported',
 'phase08_sha256':PRIOR_SHA,
 'private_reproducibility_manifest_sha256':sha256_file(PRIVATE_MANIFEST_FILE),
 'stable_private_hmac_key_reused':KEY_REUSED,
 'environment':{'python':platform.python_version(),'pandas':pd.__version__,'jsonschema':jsonschema.__version__},
 'source':{'rows':len(data),'schema_fields':len(data.columns)-2,'feature_events':len(EVIDENCE)},
 'fhir_r4_validation':SCHEMA_STATUS,
 'private_candidate_etl':{
    'person_rows':before['PERSON_CANDIDATE'],
    'image_occurrence_candidate_rows':before['IMAGE_OCCURRENCE_CANDIDATE'],
    'image_feature_candidate_rows':before['IMAGE_FEATURE_CANDIDATE'],
    'measurement_candidate_rows':before['MEASUREMENT_CANDIDATE'],
    'source_vs_fhir_event_count_difference':sum(source_event_counts.values())-len(EVIDENCE),
    'fhir_to_candidate_count_difference':count_mismatches,
    'fhir_numeric_roundtrip_decimal_mismatches':exact_value_mismatches,
    'foreign_key_failures':foreign_key_errors,
    'idempotent_repeat_import_no_additional_rows':before==after,
    'source_vs_target_structural_bilateral_hippocampus_symdiff':structural_diff,
    'structural_source_candidate_membership':len(source_membership),
    'structural_target_candidate_membership':len(target_membership),
    'official_micdm_columns_missing_actual_evidence':missing_mandatory,
    'source_measurement_units_verified':False,
    'source_dicom_uids_verified':False,
    'original_trace_state_loaded':False,
    'official_micdm_load_completed':False,
 },
 'athena_vocabulary':VOCAB,
 'gates':{
    'actual_fhir_to_private_candidate_etl_completed':True,
    'official_fhir_r4_validation_passed':SCHEMA_STATUS['validation']=='PASS',
    'candidate_count_and_referential_integrity_passed':not any(count_mismatches.values()) and foreign_key_errors==0,
    'exact_numeric_preservation':exact_value_mismatches==0,
    'original_trace_state_available':False,
    'authoritative_source_export_units_confirmed':False,
    'verified_dicom_study_and_series_uids':False,
    'approved_omop_concept_mappings':False,
    'athena_snapshot_scanned':VOCAB['snapshot_found'],
    'official_micdm_load_completed':False,
    'four_arm_comparison_completed':False,
 },
 'four_arm_comparator_scores':None,
 'next_stage_gate':'REVIEW_EXACT_ADNI_EXPORT_DICTIONARY_AND_ATHENA_CONCEPTS_AND_RECOVER_TRACE_EVIDENCE_BEFORE_OFFICIAL_MICDM_LOAD',
}
REPORT.write_text(json.dumps(REPORT_DATA,indent=2),encoding='utf-8')
con.close()
print(json.dumps({k:REPORT_DATA[k] for k in ('phase','result_status','gates')},indent=2))
print('Shareable output:',REPORT)

In [ ]:
try:
    from google.colab import files
    files.download(str(REPORT))
except ImportError:
    print('Shareable output saved:',REPORT)